# 📊 MESA Benchmark Suite — Tek Seferde Uçtan Uca Test (Run All Ready)

Bu notebook, **MESA Benchmark Sisteminin (v0.5.1+)** tüm özelliklerini (5 Bellek Katmanı, KùzuDB Multi-Hop Çizge, Çift Kademeli Değerlendirme & Cohen's Kappa, İstatistiksel Tekrarlanabilirlik) Google Colab üzerinde **tek tıkla ('Run All')** baştan sona çalıştıracak şekilde tasarlanmıştır.

**API Ücreti:** `$0.00` (Tamamen yerel Ollama + `llama3.2:3b` kullanılır).

## 1. Altyapı ve Bağımlılıkların Kurulumu

In [ ]:
!apt-get update -qq && apt-get install -y -qq zstd

import os
os.chdir('/content')
!rm -rf MESA
!git clone https://github.com/Yasou13/MESA.git

# MESA Core + Adapters Kurulumu (ollama SDK dahil)
%cd /content/MESA
!pip install -q -e ".[adapters]" ollama

# Benchmark Bağımlılıkları Kurulumu
%cd /content/MESA/mesa-benchmark
!pip install -q -r requirements.txt
print('✅ MESA, adaptörler ve ollama Python kütüphanesi başarıyla kuruldu.')


## 2. Yerel Ollama LLM Sunucusunu Başlatma

In [ ]:
import os
import time
import subprocess

os.chdir('/content/MESA/mesa-benchmark')

# Çevresel değişkenler (Sıfır Maliyet & Ollama Yönlendirmesi)
os.environ['OPENAI_BASE_URL'] = 'http://127.0.0.1:11434/v1'
os.environ['OPENAI_API_KEY'] = 'ollama-dummy-key'
os.environ['MESA_ZERO_COST_MODE'] = 'true'
os.environ['MESA_LLM_PROVIDER'] = 'ollama'
os.environ['MESA_OLLAMA_URL'] = 'http://127.0.0.1:11434'
os.environ['LLM_MODEL_NAME'] = 'llama3.2:3b'
os.environ['MESA_STORAGE_PATH'] = '/content/mesa_storage'

!curl -fsSL https://ollama.com/install.sh | sh
!nohup ollama serve > /dev/null 2>&1 &

print('⏳ Ollama sunucusu bekleniyor...')
for _ in range(15):
    try:
        res = subprocess.run(['curl', '-s', 'http://127.0.0.1:11434/api/tags'], capture_output=True, text=True)
        if 'models' in res.stdout:
            print('✅ Ollama sunucusu aktif!')
            break
    except Exception:
        pass
    time.sleep(2)
else:
    print('❌ Ollama sunucusu başlatılamadı.')

!ollama pull llama3.2:3b
print('✅ llama3.2:3b modeli hazır!')


## 3. Benchmark Yapılandırması (`config.yaml`)
Sistemi hem KùzuDB Multi-Hop hem de Çift Kademeli Değerlendirme (Keyword vs LLM-Judge) yapacak şekilde hazırlarız.

In [ ]:
import yaml
import os

os.chdir('/content/MESA/mesa-benchmark')
config_path = 'config.yaml'

with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

# Ollama Yargıç & Çift Kademeli Uyum Testi
config['evaluation']['llm_judge_model'] = 'llama3.2:3b'
config['evaluation']['enable_agreement'] = True
config['iterations'] = 1

with open(config_path, 'w') as f:
    yaml.dump(config, f)

print('✅ config.yaml güncellendi (Multi-Hop Graph + Dual-Scoring aktif)')


## 4. Ana Benchmark Testini Çalıştır
MESA hem Vektör (LanceDB) hem İlişkisel (SQLite) hem de Çizge (KùzuDB) katmanlarında senaryoları test eder.

In [ ]:
import os
os.chdir('/content/MESA/mesa-benchmark')
!rm -rf /content/MESA/mesa-benchmark/.benchmark_state /content/mesa_storage

!python -m mesa_benchmark -c config.yaml


## 5. Üretilen Markdown Detay Raporu
Doğruluk oranı, Hit@K arama başarısı, gecikme ve Cohen's Kappa uyum tablosu:

In [ ]:
import os
import glob
from IPython.display import Markdown, display

os.chdir('/content/MESA/mesa-benchmark')
reports = glob.glob('report_*.md')
if reports:
    latest_report = max(reports, key=os.path.getctime)
    print(f'📌 Okunan Rapor: {latest_report}\n')
    with open(latest_report, 'r', encoding='utf-8') as f:
        display(Markdown(f.read()))
else:
    print('⚠️ Rapor bulunamadı.')


## 6. Multi-Seed İstatistiksel Tekrarlanabilirlik Testi
Farklı rastgelelik tohumlarında (seed: 42, 43, 44) test edilerek istatistiksel varyans ve güven aralığı ölçülür.

In [ ]:
import os
os.chdir('/content/MESA')
!python scripts/reproduce_benchmark.py --config mesa-benchmark/config.yaml --seeds 42,43,44 --max-scenarios 5
